## creating customer dim table from customer table
dim table data's
- customerkey
- gender
- continent
- country
- state
- city 


In [0]:
%sql
create or replace table `03_gold`.dim_tables.dim_customer
AS 
select 
  customerkey,
  gender,
  continent,
  country,
  state,
  city 
from `02_silver`.transformation.customers

## creating products dim table from products table
dim table data's
- productkey
- product_name
- brand
- color
- category
- subcategory
- unit_price_usd


In [0]:
%sql
create or replace table `03_gold`.dim_tables.dim_products
AS 
select 
  productkey,
  product_name,
  brand,
  color,
  category,
  subcategory,
  unit_price_usd
from `02_silver`.transformation.products

%md
## creating stores dim table from stores table
dim table data's
- storekey
- country
- state
- square_meters

In [0]:
%sql
create or replace table `03_gold`.dim_tables.dim_stores
 AS
SELECT
    storekey,
    country AS store_country,
    state,
    square_meters
from `02_silver`.transformation.stores

storekey,
    country AS store_country,
    state,
    square_meters

## creating dates dim table from sales table
dim table data's
- order date
- year
- month
- date

In [0]:
%sql
create or replace table `03_gold`.dim_tables.dim_dates AS
SELECT DISTINCT
    order_date AS date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    DAY(order_date) AS day
from `02_silver`.transformation.sales
WHERE order_date IS NOT NULL;

## creating exchange dim table from exchange table
dim table data's
- date
- currency
- exchange


In [0]:
%sql

CREATE OR REPLACE TABLE `03_gold`.dim_tables.dim_exchange_rate AS
SELECT
    date,
    currency,
    exchange
FROM `02_silver`.transformation.exchange_rates;

## creating sales fact table from sales table

In [0]:
%sql
CREATE OR REPLACE TABLE `03_gold`.fact_tables.fact_sales AS
SELECT
    s.order_number,
    s.line_item,
    s.order_date,
    d.year,
    d.month,
    s.customerkey,
    s.productkey,
    s.storekey,
    s.quantity,
    p.unit_price_usd,
    er.exchange,
    ROUND(
        (s.quantity * p.unit_price_usd) / er.exchange,
        2
    ) AS revenue_usd,
    DATEDIFF(s.delivery_date, ifnull(s.order_date,s.delivery_date)) AS delivery_days,
    CASE 
        WHEN s.storekey = 0 THEN 'online'
        ELSE 'store'
    END AS channel,
    s.currency_code

FROM `02_silver`.transformation.sales as s
LEFT JOIN `03_gold`.dim_tables.dim_products as p
    ON s.productkey = p.productkey
LEFT JOIN `03_gold`.dim_tables.dim_dates d
    ON s.order_date = d.date
LEFT JOIN `03_gold`.dim_tables.dim_exchange_rate er
    ON s.currency_code = er.currency
    AND s.order_date = er.date;

In [0]:
%sql
select * from `03_gold`.fact_tables.fact_sales

In [0]:
%skip
sales = spark.read.table("`02_silver`.transformation.sales")
products = spark.read.table("`02_silver`.transformation.products")
exchange = spark.read.table("`02_silver`.transformation.exchange_rates")

In [0]:
%skip
prod_join = sales.alias("s").join(products.alias("p"), on="productkey",how="left").select("s.productkey","s.quantity","s.order_date","p.product_name","p.unit_cost_usd","p.unit_price_usd","s.currency_code")
display(prod_join)

In [0]:
%skip
from pyspark.sql import functions as F
from pyspark.sql import functions as F
exchange = prod_join.alias("p").join(
    exchange.alias("e"), 
    (F.col("p.currency_code") == F.col("e.currency")) & (F.col("p.order_date") == F.col("e.date")),
    how="left"
)

display(exchange)



In [0]:
%skip
display(
    exchange.groupBy(
        F.year("order_date").alias("year"),
        F.month("order_date").alias("month")
    )
    .agg(
        F.sum((F.col("unit_price_usd") * F.col("quantity")) /  F.col("exchange"))
         .cast("decimal(18,2)")
         .alias("total_price")
    )
    .filter(F.col("year") == 2020)
    .orderBy("year","month")
)
